# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG Chatbot (v4)**

Construct a QA Bot that Leverages LangChain and LLMs to Answer Questions from Loaded Documents

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


---

## 📋 v4 — Mejoras combinadas

| # | Componente | Origen | Descripción |
|---|-----------|--------|-------------|
| 1 | **Chain + retrieval** | v3 | `RetrievalQA` con prompt default — responde bien sin bloquear |
| 2 | **Source citation** | v3 | `📎 Sources:` al final de cada respuesta |
| 3 | **pip con sys.executable** | v3 | Siempre instala en el kernel correcto |
| 4 | **Gradio ChatInterface** | v2 | UI conversacional con historial de chat |
| 5 | **File persistence** | v2 | El PDF se mantiene entre preguntas sin re-subir |
| 6 | **Error handling** | v2+v3 | Mensaje claro si LM Studio no está corriendo |

---

🧩 **Step 1 – Install the required packages**

In [2]:
import sys

!{sys.executable} -m pip install langchain
!{sys.executable} -m pip install langchain-classic
!{sys.executable} -m pip install langchain-community
!{sys.executable} -m pip install langchain-openai
!{sys.executable} -m pip install langchain-huggingface
!{sys.executable} -m pip install chromadb
!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install gradio
!{sys.executable} -m pip install openai

---

## ⚙️ Step 2 – Imports

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

import gradio as gr
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

C:\Users\baraj\AppData\Local\Temp\ipykernel_32024\1976649715.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


---

## 🔧 Step 3 – Configure LLM pointing to LM Studio

In [4]:
LMSTUDIO_BASE_URL = "http://100.111.50.52:1234/v1"
LMSTUDIO_MODEL    = "qwen2.5-coder-7b-instruct"

def get_llm():
    return ChatOpenAI(
        base_url=LMSTUDIO_BASE_URL,
        api_key="not-needed",
        model=LMSTUDIO_MODEL,
        temperature=0.5,
        max_tokens=1024,
    )

In [5]:
# Test connection
llm = get_llm()
resp = llm.invoke("Dame una respuesta corta en español diciendo que la conexión con LM Studio funciona.")
print(resp.content)

La conexión con LM Studio funciona correctamente.


---

## 📄 Step 4 – Document loader

In [6]:
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    source_name = os.path.basename(file_path)
    for doc in docs:
        doc.metadata["source_file"] = source_name
    return docs

---

## ✂️ Step 5 – Text splitter

In [7]:
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
    )
    return splitter.split_documents(docs)

---

## 🧠 Step 6 – Embeddings + VectorDB

In [8]:
def embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def vector_database(chunks):
    embed = embedding_model()
    return Chroma.from_documents(documents=chunks, embedding=embed)

---

## 🔗 Step 7 – Retriever + QA chain with source citation

In [9]:
def build_retriever(file_paths):
    all_docs = []
    for fp in file_paths:
        all_docs.extend(document_loader(fp))
    chunks = text_splitter(all_docs)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever()


def answer_question(file_paths, question):
    llm = get_llm()
    retriever = build_retriever(file_paths)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
    )

    result = qa_chain.invoke({"query": question})
    answer = result["result"]

    # v3/v4: append source citation
    sources = set(
        doc.metadata.get("source_file", "unknown")
        for doc in result.get("source_documents", [])
    )
    if sources:
        answer += f"\n\n---\n📎 Sources: {', '.join(sorted(sources))}"

    return answer

---

## 💻 Step 8 – Gradio ChatInterface (from v2)

Uses `gr.ChatInterface` for a conversational UI with chat history.
The uploaded PDF persists across all questions in the session.

In [10]:
# Persist uploaded files across chat turns
_uploaded_files = None

def gradio_rag_v4(message, history, file):
    """
    gr.ChatInterface signature: (message, history, *additional_inputs)
    Must return a string — Gradio manages history internally.
    """
    global _uploaded_files

    # Update stored files whenever a new upload arrives
    if file is not None:
        _uploaded_files = file if isinstance(file, list) else [file]

    if not _uploaded_files:
        return "Please upload a PDF file before asking questions."

    try:
        return answer_question(_uploaded_files, message)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ Cannot connect to LM Studio. Make sure it's running at http://100.111.50.52:1234"
        return f"Error: {err}"

In [11]:
rag_app = gr.ChatInterface(
    fn=gradio_rag_v4,
    additional_inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
    ],
    title="🤖 ITESM-NLP RAG Chatbot v4",
    description=(
        "Upload a PDF and ask questions in a conversational style.\n"
        "Each answer includes the source PDF it came from."
    ),
)

rag_app.launch(server_name="127.0.0.1", server_port=7864)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


---

### Stop the server and release the port

In [1]:
gr.close_all()
rag_app.close()

NameError: name 'gr' is not defined

---

## 📝 Conclusiones — Problemas encontrados al reducir alucinaciones en RAG local

Durante el desarrollo de este chatbot RAG intentamos implementar múltiples técnicas de reducción de alucinaciones. A continuación se documentan los problemas reales que encontramos en cada intento y cómo los resolvimos.

---

### ❌ Problema 1 — Filtro de similitud con umbral demasiado alto (v2)

**Técnica aplicada:** Filtrar los chunks recuperados por ChromaDB usando un umbral de similitud coseno (`SIMILARITY_THRESHOLD = 0.35`) para descartar contexto irrelevante antes de enviarlo al LLM.

**Problema:** El modelo de embeddings `all-MiniLM-L6-v2` combinado con la métrica de distancia L2 de ChromaDB produce scores de relevancia en un rango de **0.04 a 0.09** — muy por debajo del umbral de 0.35. El resultado: **todos los chunks eran rechazados** y el chatbot respondía siempre con "no tengo información suficiente", sin importar qué PDF se subiera.

**Lección:** El rango real de scores depende del modelo de embeddings y la métrica de distancia usada. Antes de configurar un umbral, es necesario imprimir los scores crudos para conocer el rango real del sistema.

---

### ❌ Problema 2 — Prompt anti-alucinación demasiado estricto (v2 y v3)

**Técnica aplicada:** Inyectar un prompt personalizado con reglas explícitas como *"nunca uses tu conocimiento propio"*, *"cita cada afirmación con [doc:i]"* y *"di 'no sé' si el contexto no es suficiente"*.

**Problema:** El modelo local `qwen2.5-coder-7b-instruct` (7B parámetros) interpreta estas reglas **de forma demasiado literal**. Aunque el contexto recuperado contenía información relevante, el modelo evaluaba si el contexto era "suficientemente completo" y casi siempre decidía que no lo era, respondiendo con el fallback de "no tengo información". Los modelos pequeños no pueden seguir instrucciones matizadas con la misma precisión que modelos grandes (GPT-4, Claude).

**Lección:** Los prompts anti-alucinación con reglas explícitas funcionan bien con modelos grandes (+70B). Con modelos pequeños locales, la mejor estrategia de control es la **calidad del retrieval** (buenos chunks), no las instrucciones del prompt.

---

### ❌ Problema 3 — Desbordamiento del contexto (context overflow) (v2)

**Técnica aplicada:** Recuperar `k=15` chunks para maximizar el contexto disponible y mejorar la cobertura de la respuesta.

**Problema:** El modelo está cargado en LM Studio con una ventana de contexto de solo **4096 tokens**. Con 15 chunks de ~1600 caracteres cada uno (≈ 24,000 caracteres totales), el modelo recibía aproximadamente **6,000 tokens** — superando su límite y generando un `BadRequestError 400`.

**Lección:** Es necesario conocer el límite de contexto del modelo y calcular cuántos chunks caben. Con 4096 tokens, reservando 1024 para la respuesta y ~200 para el prompt, solo quedan ~2800 tokens para contexto (≈ 8,000 caracteres). Se ajustó `k=5` con un tope de 8,000 caracteres.

---

### ❌ Problema 4 — Chunks duplicados en el retrieval (v2)

**Técnica aplicada:** Recuperar múltiples chunks del PDF para enriquecer el contexto.

**Problema:** ChromaDB devolvía el **mismo chunk hasta 3 veces** dentro del top-k, inflando el contexto con información redundante y contribuyendo al desbordamiento de tokens.

**Lección:** Siempre deduplicar los chunks recuperados por contenido antes de enviarlos al LLM.

---

### ❌ Problema 5 — Conflicto de claves en `ConversationalRetrievalChain` + memoria (v2)

**Técnica aplicada:** Usar `ConversationalRetrievalChain` con `ConversationSummaryMemory` para mantener historial y reducir alucinaciones por contexto perdido entre turnos.

**Problema:** Al usar `return_source_documents=True`, la cadena retorna dos claves (`answer` y `source_documents`). `ConversationSummaryMemory` no sabe cuál almacenar y lanza: *"Got multiple output keys, cannot determine which to store in memory"*.

**Lección:** Siempre especificar `output_key="answer"` tanto en la cadena como en la memoria cuando se usan cadenas que retornan múltiples outputs.

---

### ✅ Solución final — v4

Después de todos estos intentos, la solución más efectiva resultó ser la más simple:

1. **Mantener `RetrievalQA` con el prompt default de LangChain** — el prompt default no bloquea respuestas válidas y el modelo responde naturalmente desde el contexto recuperado.
2. **Agregar citation de fuentes como post-procesamiento** — sin involucrar al LLM en la tarea de citar, simplemente extrayendo los metadatos de los documentos recuperados.
3. **Usar `gr.ChatInterface`** para una mejor experiencia conversacional.

**Conclusión general:** En sistemas RAG con modelos locales pequeños, la reducción de alucinaciones más efectiva viene de un buen pipeline de retrieval (chunks bien dimensionados, embeddings correctos, k apropiado), no de instrucciones complejas en el prompt. Las técnicas agresivas diseñadas para modelos grandes pueden empeorar el comportamiento de modelos pequeños."